# encoder
# decoder
# train
# generate

In [1]:
import torch
import torch.nn as nn

In [2]:
chars = "0123456789-/"
special_tokens = ["<SOS>", "<EOS>"]

itos = special_tokens + list(chars)
stoi = {ch: i for i, ch in enumerate(itos)}

SOS_token = stoi["<SOS>"]
EOS_token = stoi["<EOS>"]

vocab_size = len(itos)

print(itos)
print(stoi)

['<SOS>', '<EOS>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '/']
{'<SOS>': 0, '<EOS>': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, '-': 12, '/': 13}


In [3]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.0)

        # 你来写 embedding
        # 你来写 rnn

    def forward(self, input):
        # 要限定 input的shape
        # input.shape (batch_size, seqlen)
        # 你来写一次前向传播
        input = self.embedding(input)
        input = self.dropout(input)
        # hidden不受 batch_first=True的控制，其shape仍然为(direction*num,batch_size,hidden_size)
        # output.shape 是 (batch_size,seq_len,direction*hidden_size)
        output, hidden = self.rnn(input)
        return output, hidden


In [4]:
import random
def make_sample(sample_size):
    ep = sample_size
    inputs = []
    targets = []
    for i in range(ep):
        year = random.randint(1950, 2050)
        day = random.randint(1, 28)
        month = random.randint(1, 12)
        input = f"{year}-{month:02d}-{day:02d}"
        inputs.append(input)
        target = f"{day}/{month:02d}/{year:02d}"
        targets.append(target)
    return inputs, targets

inputs, targets = make_sample(100)
print(inputs[0:10])
print(targets[0:10])

print("len(inputs)== len(targets)",len(inputs)== len(targets))

['2019-05-24', '1997-01-15', '1972-10-25', '2038-02-25', '1972-06-04', '1964-12-06', '2008-01-12', '1992-10-09', '1987-03-10', '1991-10-15']
['24/05/2019', '15/01/1997', '25/10/1972', '25/02/2038', '4/06/1972', '6/12/1964', '12/01/2008', '9/10/1992', '10/03/1987', '15/10/1991']
len(inputs)== len(targets) True


In [5]:
def tensor_from_string(s, add_eos=False):
    input_tensor = torch.tensor([stoi[ch] for ch in s], dtype=torch.long)
    if add_eos:
        new_value = torch.tensor([EOS_token])
        input_tensor = torch.cat([input_tensor, new_value],dim=0)
    return input_tensor.unsqueeze(0)

def string_from_tensor(tensor):
    s = ''
    tensor = tensor.squeeze(0)
    for token in tensor:
        idx = token.item()
        if idx == EOS_token:
            break
        if idx == SOS_token:
            continue
        s += itos[token]
    return s

In [6]:

x = tensor_from_string("2002-1-23")
print(x, x.shape)
y = tensor_from_string("23/1/2002", add_eos=True)
print(y, y.shape)


tensor([[ 4,  2,  2,  4, 12,  3, 12,  4,  5]]) torch.Size([1, 9])
tensor([[ 4,  5, 13,  3, 13,  4,  2,  2,  4,  1]]) torch.Size([1, 10])


In [7]:
print(string_from_tensor(x))
print(string_from_tensor(y))

2002-1-23
23/1/2002


In [8]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)

x = tensor_from_string("2002-1-23")
print("x", x.shape)

encoder_output, encoder_hidden = encoder.forward(x);
print("encoder_output:", encoder_output.shape)
print("encoder_hidden:", encoder_hidden.shape)

x torch.Size([1, 9])
encoder_output: torch.Size([1, 9, 5])
encoder_hidden: torch.Size([1, 1, 5])


In [9]:
from torch import Tensor
from typing import Optional


class DecoderRNN(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, output_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_hidden: Tensor, target_tensor: Optional[Tensor] = None, max_len: int = 20):
        # encoder_output是encoder输出的 hiddeng向量
        # target_tensor.shape is (batch_size, seq_len)
        # foward.shape is (batch_size, seq_len,vocab_size)
        batch_size = encoder_hidden.shape[1]
        hidden = encoder_hidden
        input_token = SOS_token 
        outputs = []
        # input_token的格式是 (batch_size,seq_len)
        input_token = torch.full((batch_size, 1), SOS_token, dtype=torch.long) #设置起始词元
        loop_len = max_len
        if target_tensor is not None:
            loop_len = target_tensor.shape[1]

        for i in range(loop_len):
            logits, hidden, predictIdx = self.forward_step(hidden, input_token) 
            outputs.append(logits)

            if target_tensor is not None:
                input_token = target_tensor[:,i].unsqueeze(1)
            else :
                input_token = predictIdx.detach()
        decoder_outputs = torch.stack(tensors=outputs, dim=1)
        return decoder_outputs, hidden
    
    def forward_step(self, hidden, input_token):
        # input_token.shape是（batch_size,seq_len)
        embedded = self.embedding(input_token)
        # rnn的hidden输入是(batch_size,seq_len,hidden)
        output, hidden = self.rnn(embedded, hidden)
        # hidden不受 batch_first=True的控制，其shape仍然为(direction*num,batch_size,hidden_size)
        # output.shape 是 (batch_size,seq_len,direction*hidden_size)
        logits = self.out(hidden[-1]) # hidden[-1]表示最后一个num
        predictIdx = logits.argmax(dim = -1,keepdim= True)
        return logits, hidden, predictIdx

    

In [10]:
input_token = torch.full((1, 1), SOS_token, dtype=torch.long)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)
logits, decoder_hidden, predictIdx = decoder.forward_step(encoder_hidden, input_token)

print("decoder_hidden:", decoder_hidden.shape)
print("logits:", logits.shape)
print("predictIdx", predictIdx)


decoder_hidden: torch.Size([1, 1, 5])
logits: torch.Size([1, 14])
predictIdx tensor([[13]])


In [11]:
print(decoder.out.out_features)

14


In [12]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)

x = tensor_from_string("2002-1-23")
y = tensor_from_string("23/1/2002", add_eos=True)

encoder_output, encoder_hidden = encoder(x)
decoder_outputs, decoder_hidden = decoder(encoder_hidden, y)

print("decoder_outputs:", decoder_outputs.shape)
print("decoder_hidden:", decoder_hidden.shape)
print("target:", y.shape)


decoder_outputs: torch.Size([1, 10, 14])
decoder_hidden: torch.Size([1, 1, 5])
target: torch.Size([1, 10])


In [13]:
from torch.nn.modules.loss import CrossEntropyLoss


def train_one_sample(input_str, target_str, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    input_tensor = tensor_from_string(input_str, True) 
    target_tensor = tensor_from_string(target_str, True)
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    encoder_output, encoder_hidden = encoder(input_tensor)
    decoder_outputs, decoder_hidden = decoder(encoder_hidden, target_tensor)
    logits = decoder_outputs.reshape(-1, vocab_size)
    target = target_tensor.reshape(-1)
    loss = criterion(logits, target)
    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item()


In [14]:
hidden_size = 32
encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), lr=0.01)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=0.01)

criterion = nn.CrossEntropyLoss()

loss = train_one_sample(
    "2002-1-23",
    "23/1/2002",
    encoder,
    decoder,
    encoder_optimizer,
    decoder_optimizer,
    criterion,
)

print(loss)

2.712573766708374


In [15]:
def generate(input_str, encoder, decoder, max_len=20):
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        input_tensor = tensor_from_string(input_str)
        encoder_output, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden = decoder(encoder_hidden, None, max_len)
        # decoder_outputs.shape (batch,seq_len,vocab_size)
        pred_tokens = decoder_outputs.argmax(dim=2)
        outputs = []
        for token in pred_tokens[0]:
            if token.item() == EOS_token:
                break
            outputs.append(token)
        if len(outputs) == 0:
            return ""
        
    
    return string_from_tensor(torch.stack(outputs).unsqueeze(0))

In [16]:
fixed_samples = [
    ("2002-1-23", "23/1/2002"),
    ("1999-12-8", "8/12/1999"),
    ("2020-3-7", "7/3/2020"),
    ("1950-11-28", "28/11/1950"),
    ("2048-6-15", "15/6/2048"),
]

def eval_samples(samples, encoder, decoder, verbose=False):
    encoder.eval()
    decoder.eval()

    correct_count = 0

    for input_str, target_str in samples:
        pred = generate(input_str, encoder, decoder)
        correct = pred == target_str

        if correct:
            correct_count += 1

        if verbose:
            print("input:  ", input_str)
            print("pred:   ", pred)
            print("target: ", target_str)
            print("correct:", correct)
            print("---")

    total = len(samples)
    acc = correct_count / total

    return correct_count, total, acc


In [ ]:
import random
# 循环训练
epoch = 1000
sample_size = 20000
loss_group = sample_size//10
epoch_group = epoch//10
lr = 0.001
inputs, targets = make_sample(sample_size)
samples = list(zip(inputs, targets))
# samples = fixed_samples
hidden_size = 256

encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size,vocab_size)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), lr)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr)

criterion = nn.CrossEntropyLoss()
for j in range(epoch):
    encoder.train()
    decoder.train()
    epoch_loss = 0

    random.shuffle(samples)
    for i, (input_str, target_str) in enumerate(samples):
        loss = train_one_sample(input_str, target_str, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        epoch_loss += loss
    
    if (j + 1) % epoch_group == 0:
        print("epoch:", j + 1, ", avg_epoch_loss:", epoch_loss / len(samples))

    
            
train_correct, train_total, train_acc = eval_samples(samples, encoder, decoder, )
print("train accuracy:", train_correct, "/", len(samples), train_acc)

test_inputs, test_targets = make_sample(100)
test_samples = list(zip(test_inputs, test_targets))
test_correct, test_total, test_acc = eval_samples( test_samples, encoder, decoder)
print("test accuracy:", test_correct, "/", len(test_samples), test_acc)

In [ ]:
            
from sympy import true


train_correct, train_total, train_acc = eval_samples(samples, encoder, decoder, true)
print("train accuracy:", train_correct, "/", len(samples), train_acc)

test_inputs, test_targets = make_sample(100)
test_samples = list(zip(test_inputs, test_targets))
test_correct, test_total, test_acc = eval_samples( test_samples, encoder, decoder, true)
print("test accuracy:", test_correct, "/", len(test_samples), test_acc)

input:   1998-12-12
pred:    12/12/1976
target:  12/12/1998
correct: False
---
input:   1990-4-14
pred:    14/4/1990
target:  14/4/1990
correct: True
---
input:   2047-3-18
pred:    18/3/2047
target:  18/3/2047
correct: True
---
input:   2025-11-1
pred:    1/11/2025
target:  1/11/2025
correct: True
---
input:   2010-10-25
pred:    25/10/2010
target:  25/10/2010
correct: True
---
input:   1956-10-10
pred:    10/10/1956
target:  10/10/1956
correct: True
---
input:   1960-3-11
pred:    11/3/1960
target:  11/3/1960
correct: True
---
input:   1999-12-19
pred:    19/12/1999
target:  19/12/1999
correct: True
---
input:   1981-10-21
pred:    21/10/1981
target:  21/10/1981
correct: True
---
input:   1963-9-9
pred:    9/9/1963
target:  9/9/1963
correct: True
---
input:   2049-4-28
pred:    28/4/2049
target:  28/4/2049
correct: True
---
input:   2017-9-15
pred:    15/4/1970
target:  15/9/2017
correct: False
---
input:   1967-11-7
pred:    7/11/1967
target:  7/11/1967
correct: True
---
input:   20